In [ ]:
!pip install mlflow

**All Required Commands to install MLflow on EC2**

In [ ]:
# Test mlflow
import mlflow
mlflow.set_tracking_uri("http://ec2-52-204-122-132.compute-1.amazonaws.com:5000/")
with mlflow.start_run():
  mlflow.log_param("param1",15)
  mlflow.log_metric("metric1",0.89)

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df=pd.read_csv('https://raw.githubusercontent.com/Himanshu-1703/reddit-sentiment-analysis/main/data/reddit.csv')
df.head()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df=df[~(df['clean_comment'].str.strip()=='')]

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
# Ensure that data is downloaded
nltk.download("stopwords")
nltk.donwload("wordnet")

In [ ]:
# Define the preprocessing function
def preprocess_comment(comment):
  comment=comment.lower()
  comment=comment.strip()
  comment=re.sub(r'\n',' ',comment)
  comment=re.sub(r'[^A-Za-z0-9\s!?.,]','',comment)

  # Remove stopwords but retrain important ones for sentiment analysis
  stop_words=set(stopwords.words('english')-{'not','but','however','no','yet'})
  comment=' '.join([word for word in comment.split() if word not in stopwords])

  # Lemmatize the words
  lemmatizer=WordNetLemmatizer()
  comment=' '.join([lemmatizer.lemmatize(word) for word in comment.split()])

  return comment

In [ ]:
# Apply the preprocessing function to the 'clean_comment' column
df['clean_comment']=df['clean_comment'].apply(preprocess_comment)

In [ ]:
df.head()

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StartifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Step 1: Vectorize the comments using Bag of Words (CountVectorizer)
vectorizer=CountVectorizer(max_features=10000)

In [ ]:
X=vectorizer.fit_transform(df['clean_comment']).toarray()
y=df['category']

In [ ]:
X

In [ ]:
X.shape

In [ ]:
y

In [ ]:
y.shape

In [ ]:
mlflow.set_tracking_uri("http://ec2-52-204-122-132.compute-1.amazonaws.com:5000/")

In [ ]:
# Step 2: Set up the MLflow tracking server
mlflow.set_experiment("RF Baseline")

In [ ]:
!pip install boto3

In [ ]:
!pip install awscli

In [ ]:
# AWS Access Key ID
!aws configure

In [ ]:
# Step 1: Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test=train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)

# Step 2: Define and train a Random Forest baseline model using a simple train test split
with mlflow.start_run( ) as run:
  # Log a description for the run
  mlflow.set_tag("mlflow.runName", "RandomForest_Baseline_TrainTestSplit")
  mlflow.set_tag("experiment_type","baseline")
  mlflow.set_tag("model_type", "RandomForestClassifier")

  # Add a description
  mlflow.set_tag("description","Base RandomForest model for sentiment analysis using Bag of Words (BoW)")

  # Log parameters for the vectorizer
  mlflow.log_param("vectorizer_type","CountVectorizer")
  mlflow.log_param("vectorizer_max_features", vectorizer.max_features)

  # Log Random Forest parameters
  n_estimators= 200
  max_depth= 15

  mlflow.log_param("n_estimators", n_estimators)
  mlflow.log_param("max_depth", max_depth)
  mlflow.log_param("num_features", X.shape[1])

  # Initialize and train the model
  model=RandomForestClassifier(n_estimators=n_estimators,max_depth=max_depth, random_state=42)
  model.fit(X_train,y_train)

  # Make prediction on the test set
  y_pred= model.predict(X_test)

  # Log metrics for each class and accuracy
  accuracy=accuracy_score(y_test,y_pred)
  mlflow.log_metric("accuracy",accuracy)

  classification_rep=classification_report(y_test,y_pred,out_dict=True)

  for label, metrics in classification_rep.items():
    if isinstance(metrics,dict):
      for metric, value in metrics.items():
        mlflow.log_metric(f"{label}_{metric}",value)

  # Confusion matrix plot
  conf_matrix=confusion_matrix(y_test,y_pred)
  plt.figure(figsize=(8,6))
  sns.heatmap(conf_matrix,annot=True,fmt="d",cmap='Blues')
  plt.xlabel("Predicted")
  plt.ylabel("Actual")
  plt.title("Confusion Matrix")

  # Save and log the confusion matrix plot
  plt.savefig("confusion_matrix.png")
  mlflow.log_artifact("confusion_matrix.png")

  # Log the Random Forest model
  mlflow.sklearn.log_model(model,"random_forest_model")

  # Log the dataset itself (if it's small enough)
  df.to_csv("dataset.csv",index=False)
  mlflow.log_artifact("dataset.csv")

# Display final accuracy
print(f"Accuracy: {accuracy}")

In [ ]:
print(classification_report(y_test,y_pred))

In [ ]:
df.to_csv("reddit_preprocessing.csv",index=False)

In [ ]:
pd.read_csv("reddit_preprocessing.csv").head()